# Feature Engineering

## Objective

Build a feature set for store-level weekly sales forecasting using only lag/rolling/calendar features that would be genuinely available at forecast time — no future information. Based on 03_time_series_analysis.ipynb: weak trend, holiday seasonality concentrated in specific weeks (not a uniform IsHoliday effect), and Type/Size as meaningful store-level differentiators.

In [2]:
import pandas as pd

merged = pd.read_csv("../data/processed/merged_store_dept_week.csv", parse_dates=["Date"])

store_week = merged.groupby(["Store", "Date"]).agg(
    Weekly_Sales=("Weekly_Sales", "sum"),
    IsHoliday=("IsHoliday", "first"),
    Type=("Type", "first"),
    Size=("Size", "first"),
    Temperature=("Temperature", "first"),
    Fuel_Price=("Fuel_Price", "first"),
    CPI=("CPI", "first"),
    Unemployment=("Unemployment", "first"),
    MarkDown1=("MarkDown1", "sum"),
    MarkDown2=("MarkDown2", "sum"),
    MarkDown3=("MarkDown3", "sum"),
    MarkDown4=("MarkDown4", "sum"),
    MarkDown5=("MarkDown5", "sum")
).reset_index()

store_week = store_week.sort_values(["Store", "Date"]).reset_index(drop=True)
print(store_week.shape)

(6435, 15)


## Calendar Features

In [5]:
store_week["Year"] = store_week["Date"].dt.year
store_week["WeekOfYear"] = store_week["Date"].dt.isocalendar().week.astype(int)
store_week["Month"] = store_week["Date"].dt.month

print(store_week[["Date","Year","WeekOfYear","Month"]].head())

        Date  Year  WeekOfYear  Month
0 2010-02-05  2010           5      2
1 2010-02-12  2010           6      2
2 2010-02-19  2010           7      2
3 2010-02-26  2010           8      2
4 2010-03-05  2010           9      3


## Specific Holiday Feature

Per the time-series analysis findings, a single IsHoliday boolean treats all four holidays as equivalent, which the data contradicts. Use the known Kaggle holiday weeks to create a specific holiday identity feature instead.

In [8]:
holiday_weeks = {
    "2010-02-12": "SuperBowl", "2011-02-11": "SuperBowl", "2012-02-10": "SuperBowl",
    "2010-09-10": "LaborDay", "2011-09-09": "LaborDay", "2012-09-07": "LaborDay",
    "2010-11-26": "Thanksgiving", "2011-11-25": "Thanksgiving", "2012-11-23": "Thanksgiving",
    "2010-12-31": "Christmas", "2011-12-30": "Christmas", "2012-12-28": "Christmas"
}
holiday_map = {pd.Timestamp(k): v for k, v in holiday_weeks.items()}

store_week["HolidayName"] = store_week["Date"].map(holiday_map).fillna("None")

print(store_week["HolidayName"].value_counts())

HolidayName
None            5985
SuperBowl        135
LaborDay         135
Thanksgiving      90
Christmas         90
Name: count, dtype: int64


## Lag and Rolling Features

**Leakage check:** All lag/rolling features below are computed per-store, sorted by date, using only .shift() with a positive lag before any rolling window — meaning every value at week T uses only weeks strictly before T. No feature here can see its own or a future week's Weekly_Sales.

In [11]:
store_week = store_week.sort_values(["Store", "Date"])

store_week["lag_1"] = store_week.groupby("Store")["Weekly_Sales"].shift(1)
store_week["lag_2"] = store_week.groupby("Store")["Weekly_Sales"].shift(2)
store_week["lag_52"] = store_week.groupby("Store")["Weekly_Sales"].shift(52)

store_week["rolling_mean_4"] = store_week.groupby("Store")["Weekly_Sales"].shift(1).rolling(4).mean().reset_index(level=0, drop=True)
store_week["rolling_std_4"] = store_week.groupby("Store")["Weekly_Sales"].shift(1).rolling(4).std().reset_index(level=0, drop=True)

print(store_week[["Store","Date","Weekly_Sales","lag_1","lag_2","lag_52","rolling_mean_4"]].head(10))

   Store       Date  Weekly_Sales       lag_1       lag_2  lag_52  \
0      1 2010-02-05    1643690.90         NaN         NaN     NaN   
1      1 2010-02-12    1641957.44  1643690.90         NaN     NaN   
2      1 2010-02-19    1611968.17  1641957.44  1643690.90     NaN   
3      1 2010-02-26    1409727.59  1611968.17  1641957.44     NaN   
4      1 2010-03-05    1554806.68  1409727.59  1611968.17     NaN   
5      1 2010-03-12    1439541.59  1554806.68  1409727.59     NaN   
6      1 2010-03-19    1472515.79  1439541.59  1554806.68     NaN   
7      1 2010-03-26    1404429.92  1472515.79  1439541.59     NaN   
8      1 2010-04-02    1594968.28  1404429.92  1472515.79     NaN   
9      1 2010-04-09    1545418.53  1594968.28  1404429.92     NaN   

   rolling_mean_4  
0             NaN  
1             NaN  
2             NaN  
3             NaN  
4    1.576836e+06  
5    1.554615e+06  
6    1.504011e+06  
7    1.469148e+06  
8    1.467823e+06  
9    1.477864e+06  


In [13]:
# Manually verify lag_1 for one store is genuinely the prior week, not the same week
check = store_week[store_week["Store"] == 1][["Date","Weekly_Sales","lag_1"]].head(5)
print(check)
print()
print("Manual check: row 2's lag_1 should equal row 1's Weekly_Sales:")
print("Row 1 Weekly_Sales:", store_week[store_week['Store']==1]['Weekly_Sales'].iloc[0])
print("Row 2 lag_1:", store_week[store_week['Store']==1]['lag_1'].iloc[1])

        Date  Weekly_Sales       lag_1
0 2010-02-05    1643690.90         NaN
1 2010-02-12    1641957.44  1643690.90
2 2010-02-19    1611968.17  1641957.44
3 2010-02-26    1409727.59  1611968.17
4 2010-03-05    1554806.68  1409727.59

Manual check: row 2's lag_1 should equal row 1's Weekly_Sales:
Row 1 Weekly_Sales: 1643690.9
Row 2 lag_1: 1643690.9


## Finalizing the Feature Set and Chronological Split

**Observed:** lag_52 requires 52 prior weeks per store, so the first year of each store's series can't have a complete feature row.

**Decision:** Drop rows where lag_52 is NaN rather than imputing a fabricated "prior year" value.

**Why:** There is no honest way to fill in a year-ago sales figure that doesn't exist. Dropping these rows sacrifices some early training data but keeps every remaining row genuinely usable — this is a standard, defensible tradeoff for lag-52 features, not a shortcut.

In [17]:
before = len(store_week)
model_data = store_week.dropna(subset=["lag_52"]).reset_index(drop=True)
after = len(model_data)

print("Rows before:", before)
print("Rows after dropping incomplete lag_52:", after)
print("Rows dropped:", before - after)
print()
print("Date range remaining:", model_data["Date"].min(), "to", model_data["Date"].max())

Rows before: 6435
Rows after dropping incomplete lag_52: 4095
Rows dropped: 2340

Date range remaining: 2011-02-04 00:00:00 to 2012-10-26 00:00:00


**Decision:** Split chronologically by date, not randomly — train on the earliest ~70%, validate on the next ~15%, test on the final ~15%. This mirrors how the model would actually be used (train on past, forecast future) and avoids the leakage that random splitting would cause in a time series.

In [20]:
dates_sorted = sorted(model_data["Date"].unique())
n = len(dates_sorted)

train_end = dates_sorted[int(n * 0.70)]
val_end = dates_sorted[int(n * 0.85)]

train_set = model_data[model_data["Date"] <= train_end]
val_set = model_data[(model_data["Date"] > train_end) & (model_data["Date"] <= val_end)]
test_set = model_data[model_data["Date"] > val_end]

print("Train:", train_set["Date"].min(), "to", train_set["Date"].max(), "-", len(train_set), "rows")
print("Val:  ", val_set["Date"].min(), "to", val_set["Date"].max(), "-", len(val_set), "rows")
print("Test: ", test_set["Date"].min(), "to", test_set["Date"].max(), "-", len(test_set), "rows")

Train: 2011-02-04 00:00:00 to 2012-04-20 00:00:00 - 2880 rows
Val:   2012-04-27 00:00:00 to 2012-07-27 00:00:00 - 630 rows
Test:  2012-08-03 00:00:00 to 2012-10-26 00:00:00 - 585 rows


In [22]:
print("Train/Val overlap:", set(train_set["Date"]) & set(val_set["Date"]))
print("Val/Test overlap:", set(val_set["Date"]) & set(test_set["Date"]))

Train/Val overlap: set()
Val/Test overlap: set()


In [24]:
model_data.to_csv("../data/processed/store_week_features.csv", index=False)
print("Saved:", model_data.shape)

Saved: (4095, 24)
